# core

> HTMX v4 support for FastHTML

In [ ]:
#| default_exp core

In [ ]:
#| export
import json

from fasthtml.common import *
from fasthtml.starlette import *
from fasthtml.core import *

from fastcore.basics import patch
from fastcore.utils import *
from fastcore.xml import *
from fastcore.meta import delegates




In [ ]:
#| export
@delegates(ft_hx)
def Partial(*args, **kwargs): return ft_hx("hx-partial")(*args, **kwargs)


In [ ]:
#| export
htmx4src   = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha6/dist/htmx.js")

In [ ]:
#| export
# When htmx4=True, configures htmx v4 with metaCharacter="-"
def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx and htmx4: raise ValueError("Cannot enable both htmx and htmx4")
    if htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    if htmx4: 
        # metaCharacter="-" makes htmx4 use dashes instead of colons (Python-friendly)
        meta_cfg = Meta(name="htmx:config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src,fhjsscr] + hdrs 
    # TODO: Check if fhjsscr works with htmx4
    return [charset, viewport] + hdrs

In [ ]:
#| export
def _wrap_ex(f, status_code, hdrs, ftrs, htmlkw, bodykw, body_wrap):
    "Wrap exception handler with FastHTML request processing"
    async def _f(req, exc):
        req.hdrs,req.ftrs,req.htmlkw,req.bodykw = map(deepcopy, (hdrs, ftrs, htmlkw, bodykw))
        req.body_wrap = body_wrap
        res = await _handle(f, req, exc)
        return _resp(req, res, status_code=status_code)
    return _f

In [ ]:
#| export
def _list(o):
    "Wrap non-list item in a list, returning empty list if None"
    return [] if not o else list(o) if isinstance(o, (tuple,list)) else [o]

In [ ]:
#| export
# Patch FastHTML.__init__ to add htmx4 support
# - Adds `htmx4=False` parameter to toggle htmx v4 headers
# - Passes htmx4 to def_hdrs() which handles the header selection
@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
                secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware,before,after = map(_list, (middleware,before,after))
    self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
    hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
    exts = {k:htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr)
        from IPython.display import display,HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
    self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
    self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                            max_age=max_age, path=sess_path, same_site=same_site,
                            https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
    super(FastHTML, self).__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)

In [ ]:
#| export
def _get_tbl(dt, nm, schema):
    render = schema.pop('render', None)
    tbl = dt[nm]
    if tbl not in dt: tbl.create(**schema)
    else: tbl.create(**schema, transform=True)
    dc = tbl.dataclass()
    if render: dc.__ft__ = render
    return tbl,dc

def _app_factory(*args, **kwargs) -> FastHTML | FastHTMLWithLiveReload:
    "Creates a FastHTML or FastHTMLWithLiveReload app instance"
    if kwargs.pop('live', False): return FastHTMLWithLiveReload(*args, **kwargs)
    kwargs.pop('reload_attempts', None)
    kwargs.pop('reload_interval', None)
    return FastHTML(*args, **kwargs)

# Supports htmx4=True for htmx v4 compatibility
def fast_app(
        db_file:Optional[str]=None, # Database file name, if needed
        render:Optional[callable]=None, # Function used to render default database class
        hdrs:Optional[tuple]=None, # Additional FT elements to add to <HEAD>
        ftrs:Optional[tuple]=None, # Additional FT elements to add to end of <BODY>
        tbls:Optional[dict]=None, # Experimental mapping from DB table names to dict table definitions
        before:Optional[tuple]|Beforeware=None, # Functions to call prior to calling handler
        middleware:Optional[tuple]=None, # Standard Starlette middleware
        live:bool=False, # Enable live reloading
        debug:bool=False, # Passed to Starlette, indicating if debug tracebacks should be returned on errors
        title:str="FastHTML page", # Default page title
        routes:Optional[tuple]=None, # Passed to Starlette
        exception_handlers:Optional[dict]=None, # Passed to Starlette
        on_startup:Optional[callable]=None, # Passed to Starlette
        on_shutdown:Optional[callable]=None, # Passed to Starlette
        lifespan:Optional[callable]=None, # Passed to Starlette
        default_hdrs=True, # Include default FastHTML headers such as HTMX script?
        pico:Optional[bool]=None, # Include PicoCSS header?
        surreal:Optional[bool]=True, # Include surreal.js/scope headers?
        htmx:Optional[bool]=True, # Include HTMX header?
        htmx4:Optional[bool]=False, # Include HTMX4 header?
        exts:Optional[list|str]=None, # HTMX extension names to include
        canonical:bool=True, # Automatically include canonical link?
        secret_key:Optional[str]=None, # Signing key for sessions
        key_fname:str='.sesskey', # Session cookie signing key file name
        session_cookie:str='session_', # Session cookie name
        max_age:int=365*24*3600, # Session cookie expiry time
        sess_path:str='/', # Session cookie path
        same_site:str='lax', # Session cookie same site policy
        sess_https_only:bool=False, # Session cookie HTTPS only?
        sess_domain:Optional[str]=None, # Session cookie domain
        htmlkw:Optional[dict]=None, # Attrs to add to the HTML tag
        bodykw:Optional[dict]=None, # Attrs to add to the Body tag
        reload_attempts:Optional[int]=1, # Number of reload attempts when live reloading
        reload_interval:Optional[int]=1000, # Time between reload attempts in ms
        static_path:str=".",  # Where the static file route points to, defaults to root dir
        body_wrap:callable=noop_body, # FT wrapper for body contents
        nb_hdrs:bool=False, # If in notebook include headers inject headers in notebook DOM?
        **kwargs):
    "Create a FastHTML or FastHTMLWithLiveReload app."
    h = (picolink,) if pico or (pico is None and default_hdrs) else ()
    if hdrs: h += tuple(hdrs)

    app = _app_factory(hdrs=h, ftrs=ftrs, before=before, middleware=middleware, live=live, debug=debug, title=title, routes=routes, exception_handlers=exception_handlers,
                  on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan, default_hdrs=default_hdrs, secret_key=secret_key, canonical=canonical,
                  session_cookie=session_cookie, max_age=max_age, sess_path=sess_path, same_site=same_site, sess_https_only=sess_https_only,
                  sess_domain=sess_domain, key_fname=key_fname, exts=exts, surreal=surreal, htmx=htmx, htmx4=htmx4, htmlkw=htmlkw,
                  reload_attempts=reload_attempts, reload_interval=reload_interval, body_wrap=body_wrap, nb_hdrs=nb_hdrs, **(bodykw or {}))
    app.static_route_exts(static_path=static_path)
    if not db_file: return app,app.route

    db = database(db_file)
    if not tbls: tbls={}
    if kwargs:
        if isinstance(first(kwargs.values()), dict): tbls = kwargs
        else:
            kwargs['render'] = render
            tbls['items'] = kwargs
    dbtbls = [_get_tbl(db.t, k, v) for k,v in tbls.items()]
    if len(dbtbls)==1: dbtbls=dbtbls[0]
    return app,app.route,*dbtbls

# WS

Below is the experiment to support WS htmx4 for fasthtml

Action normally needed: 
htmx v4 WebSocket changes from v2:

- Receiving: Form data moved from top-level to values object, HEADERS → headers (lowercase)
- Sending: Expects JSON envelope with channel, format, target, swap, payload instead of raw HTML

Patches needed:

- _find_wsp: Look in data['values'] instead of data for form fields
- _send_ws: Build JSON envelope instead of sending raw HTML

In [ ]:
from inspect import Parameter
empty = Parameter.empty

In [ ]:
def _get_htmx(h):
    res = {k:h.get(v.lower(), None) for k,v in htmx_hdrs.items()}
    return HtmxHeaders(**res)

In [ ]:
def _fix_anno(t, o):
    "Create appropriate callable type for casting a `str` to type `t` (or first type in `t` if union)"
    origin = get_origin(t)
    if origin is Union or origin is UnionType or origin in (list,List):
        t = first(o for o in get_args(t) if o!=type(None))
    d = {bool: str2bool, int: str2int, date: str2date, UploadFile: noop}
    res = d.get(t, t)
    if origin in (list,List): return _mk_list(res, o)
    if not isinstance(o, (str,list,tuple)): return o
    return res(o[-1]) if isinstance(o,(list,tuple)) else res(o)

In [ ]:
def _find_wsp_patch(ws, data, hdrs, arg:str, p:Parameter):
    "In `data` find param named `arg` of type in `p` (`arg` is ignored for body types)"
    anno = p.annotation
    if isinstance(anno, type):
        if issubclass(anno, HtmxHeaders): return _get_htmx(hdrs)
        if issubclass(anno, Starlette): return ws.scope['app']
        if issubclass(anno, WebSocket): return ws
        if issubclass(anno, dict): return data
    if anno is empty:
        if arg.lower()=='ws': return ws
        if arg.lower()=='scope': return dict2obj(ws.scope)
        if arg.lower()=='data': return data
        if arg.lower()=='htmx': return _get_htmx(hdrs)
        if arg.lower()=='app': return ws.scope['app']
        if arg.lower()=='send': return partial(_send_ws, ws)
        if 'session'.startswith(arg.lower()): return ws.scope.get('session', {})
        return None
    res = data.get('values', {}).get(arg, None)  # htmx v4: form fields in `values`
    if res is empty or res is None: res = hdrs.get(arg, None)
    if res is empty or res is None: res = p.default
    if not isinstance(res, (list,str)) or anno is empty: return res
    return [_fix_anno(anno, o) for o in res] if isinstance(res,list) else _fix_anno(anno, res)

In [ ]:
async def _send_ws(ws, resp, target=None, swap=None, channel="ui", format="html", request_id=None):
    "Send WebSocket message in htmx v4 JSON envelope format"
    if not resp: return
    payload = to_xml(resp, indent=fh_cfg.indent)
    msg = dict(channel=channel, format=format, payload=payload, target=target, swap=swap, request_id=request_id)
    await ws.send_text(json.dumps(msg))

# Apply patches
# import fasthtml.core as _core
# _core._find_wsp = _find_wsp_patch
# _core._send_ws = _send_ws

Can you summarize for me what should I do to adapt ws htmx v4 to fasthtml. I have given above some code to makes ws hx4 works but it's when I haven't thought about integrating to this hx4_patch yet so probably need to make it cleaner

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at your code, here's what you need to do to properly integrate WebSocket htmx v4 support into your `hx4_patch`:

## Key Changes Needed

**1. Add the WebSocket script to headers**
You already have `ws4src` defined but need to include it when `htmx4=True`. Update your `def_hdrs` function to optionally add it.

**2. Patch the WebSocket functions**
You have two key patches ready:
- `_find_wsp_patch` - looks for form data in `data['values']` instead of top-level `data`
- `_send_ws` - wraps responses in JSON envelope format with `channel`, `format`, `payload`, etc.

**3. Apply the patches**
You commented out the patch application at the bottom. You'll need to uncomment and adapt it to work with your module structure.

**4. Consider the API**
Your `_send_ws` now requires `target` and `swap` parameters. The old version used `hx_swap_oob`. You'll need to decide if you want to:
- Keep the new explicit parameter approach, or
- Auto-detect from the FT element's attributes

Would you like me to show you how to integrate the WebSocket script into `def_hdrs`, or would you prefer to start with applying the patches first?

In the current version with htmx 2, when do the code include the ws script?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In htmx 2, the WebSocket script is included when you pass `exts='ws'` (or `exts=['ws']`) to `FastHTML()` or `fast_app()`.

Looking at your current code, the `def_hdrs` function handles this:

```python
exts = {k:htmx_exts[k] for k in exts}
# ...
hdrs += [Script(src=ext) for ext in exts.values()]
```

So `htmx_exts['ws']` contains the URL to the htmx 2 WebSocket extension script, and it gets added to the headers.

For htmx 4, you'd want similar behavior - when someone passes `exts='ws'` (or similar), it should include your `ws4src` script instead of the v2 one.

Does that make sense? Would you like to explore how to handle this for v4?

In [ ]:
htmx_exts

{'morph': 'https://cdn.jsdelivr.net/npm/idiomorph@0.7.3/dist/idiomorph-ext.min.js',
 'head-support': 'https://cdn.jsdelivr.net/npm/htmx-ext-head-support@2.0.4/head-support.js',
 'preload': 'https://cdn.jsdelivr.net/npm/htmx-ext-preload@2.1.1/preload.js',
 'class-tools': 'https://cdn.jsdelivr.net/npm/htmx-ext-class-tools@2.0.1/class-tools.js',
 'loading-states': 'https://cdn.jsdelivr.net/npm/htmx-ext-loading-states@2.0.1/loading-states.js',
 'multi-swap': 'https://cdn.jsdelivr.net/npm/htmx-ext-multi-swap@2.0.0/multi-swap.js',
 'path-deps': 'https://cdn.jsdelivr.net/npm/htmx-ext-path-deps@2.0.0/path-deps.js',
 'remove-me': 'https://cdn.jsdelivr.net/npm/htmx-ext-remove-me@2.0.0/remove-me.js',
 'debug': 'https://unpkg.com/htmx.org@1.9.12/dist/ext/debug.js',
 'ws': 'https://cdn.jsdelivr.net/npm/htmx-ext-ws@2.0.3/ws.js',
 'chunked-transfer': 'https://cdn.jsdelivr.net/npm/htmx-ext-transfer-encoding-chunked@0.4.0/transfer-encoding-chunked.js'}

In [ ]:
htmx_exts['ws4'] = 'https://unpkg.com/htmx.org@4.0.0-alpha6/dist/ext/hx-ws.js'

Maybe I can do something that extract use the URL of ws4 if in FastHTML we use ext = ws or ws4

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good idea! You could handle both `'ws'` and `'ws4'` in the extensions, where:
- `'ws'` → uses the htmx 2 WebSocket extension (current behavior)
- `'ws4'` → uses the htmx 4 WebSocket extension (your new URL)

This would let users explicitly choose which version they want. You'd modify the extension handling logic in `def_hdrs` or wherever extensions are processed to check if `htmx4=True` and `'ws'` is in exts, then automatically use `'ws4'` instead.

Or even simpler: when `htmx4=True`, you could automatically map `'ws'` → `'ws4'` so existing code doesn't need to change.

Which approach sounds better to you?

I want if user choose either ws or ws4, we map it to ws4

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good plan! So when `htmx4=True`, both `'ws'` and `'ws4'` should map to the htmx 4 WebSocket extension.

You'll need to add this mapping logic somewhere in the initialization. The best place would be in the `FastHTML.__init__` patch or in `fast_app`, before the extensions are processed.

Something like:
```python
if htmx4 and exts:
    exts = ['ws4' if e in ('ws', 'ws4') else e for e in exts]
```

Would you like to try adding this yourself, or would you like me to show you exactly where to put it in your code?

In [ ]:
# Patch FastHTML.__init__ to add htmx4 support
# - Adds `htmx4=False` parameter to toggle htmx v4 headers
# - Passes htmx4 to def_hdrs() which handles the header selection
# - Maps 'ws' and 'ws4' extensions to 'ws4' when htmx4=True
@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
                secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware,before,after = map(_list, (middleware,before,after))
    self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
    hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
    if htmx4 and exts:
        exts = ['ws4' if e in ('ws', 'ws4') else e for e in exts]
    exts = {k:htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr)
        from IPython.display import display,HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
    self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
    self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                            max_age=max_age, path=sess_path, same_site=same_site,
                            https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
    super(FastHTML, self).__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)

Is it ok?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that looks good! You've added the mapping logic in the right place - right after `exts` is converted to a list and before it's used to look up the extension URLs.

Now you need to apply the WebSocket patches (`_find_wsp_patch` and `_send_ws`) so that the WebSocket handler actually works with the v4 format. 

Would you like to work on applying those patches next?